<a href="https://colab.research.google.com/github/guzonghua/saavlabs/blob/main/Lab3/Lab3_Highway_Env.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC144V Lab 3: Decision Making for Highway Driving with Deep Q-Learning

Group name: **TODO**   Members: **TODO**

In this lab you control a car on a simulated highway with [highway-env](https://github.com/eleurent/highway-env)
(Gymnasium API) and train a Deep Q-Network agent with [Stable-Baselines3](https://github.com/DLR-RM/stable-baselines3).
You will (1) explore the environment as a Markov Decision Process, (2) write a simple rule-based driver as a baseline,
(3) train a DQN agent, and (4) change the reward function (add a penalty for changing lanes) and observe how the driving behaviour changes.

Prerequisites: L9.1 Markov Decision Process, L9.2 Q-Learning. A GPU is **not** needed (the network is a small MLP;
use the default CPU runtime). Training takes about 10 minutes per run, so start the training cells early.

| Task | Points |
|---|---|
| 1. Explore the environment | 15 |
| 2. Rule-based baseline | 15 |
| 3. Train a DQN agent | 25 |
| 4. Reward shaping: lane-change penalty | 40 |
| Report clarity | 5 |

## Setup

In [ ]:
%pip -q install highway-env stable-baselines3

import os, time
os.environ['SDL_VIDEODRIVER'] = 'offscreen'       # headless rendering with pygame ('dummy' gives black frames)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import gymnasium as gym, highway_env
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from PIL import Image
from IPython.display import Image as IPyImage, display

ENV_ID = 'highway-fast-v0'      # lighter variant of 'highway-v0' (3 lanes, 20 vehicles, coarser simulation step): ~10x faster
ACTIONS = {0: 'LANE_LEFT', 1: 'IDLE', 2: 'LANE_RIGHT', 3: 'FASTER', 4: 'SLOWER'}
print('highway-env', highway_env.__version__, '| gymnasium', gym.__version__)

class LaneChangePenalty(gym.Wrapper):
    """Adds `penalty` (a negative number) to the reward whenever the ego vehicle starts a lane change.
    The wrapper also puts info['lane_changed'] (True/False) into the info dict. Used in Task 4."""
    def __init__(self, env, penalty=0.0):
        super().__init__(env); self.penalty = penalty
    def _target_lane(self): return self.env.unwrapped.vehicle.target_lane_index[2]
    def reset(self, **kwargs):
        out = self.env.reset(**kwargs); self._lane = self._target_lane(); return out
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        lane = self._target_lane(); changed = lane != self._lane; self._lane = lane
        info['lane_changed'] = changed
        return obs, reward + (self.penalty if changed else 0.0), terminated, truncated, info

def make_env(config=None, render=False, lane_penalty=0.0):
    """Create the environment. `config` is a dict that overrides entries of the default configuration;
    `lane_penalty` (<= 0) is the reward added at every lane change."""
    env = gym.make(ENV_ID, config=config or {}, render_mode='rgb_array' if render else None)
    return LaneChangePenalty(env, lane_penalty)

def decode_obs(obs):
    """Kinematics observation (5 x 5, normalised) -> physical units.
    Returns (ego_speed in m/s, others) where `others` is an array with one row [dx (m), dy (m), dvx (m/s)] per visible
    vehicle, relative to the ego vehicle: dx > 0 means ahead of us, |dy| < 2 m means in our lane (lane width 4 m)."""
    ego_speed = 80.0 * obs[0, 3]
    o = obs[1:][obs[1:, 0] > 0.5]
    others = np.stack([200.0 * o[:, 1], 12.0 * o[:, 2], 80.0 * o[:, 3]], axis=1) if len(o) else np.zeros((0, 3))
    return ego_speed, others

def evaluate(policy, n_episodes=30, config=None, seed0=1000):
    """Run `policy(obs) -> action` for n_episodes episodes (fixed seeds, so results are comparable).
    Returns a dict of averages: ret (return under `config`), crash_rate, speed (m/s), length (steps), lane_changes."""
    env = make_env(config); rows = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed0 + ep)
        done, R, speeds, L, lane_changes = False, 0.0, [], 0, 0
        lane = env.unwrapped.vehicle.lane_index[2]
        while not done:
            obs, r, terminated, truncated, info = env.step(int(policy(obs)))
            R += r; L += 1; speeds.append(info['speed']); done = terminated or truncated
            new_lane = env.unwrapped.vehicle.lane_index[2]; lane_changes += int(new_lane != lane); lane = new_lane
        rows.append((R, float(info['crashed']), np.mean(speeds), L, lane_changes))
    a = np.array(rows)
    return dict(ret=a[:, 0].mean(), crash_rate=a[:, 1].mean(), speed=a[:, 2].mean(), length=a[:, 3].mean(), lane_changes=a[:, 4].mean())

def rollout_gif(policy, path='episode.gif', config=None, seed=0, max_steps=60):
    """Record one episode as an animated GIF (one frame per decision step) and display it."""
    env = make_env(config, render=True); obs, _ = env.reset(seed=seed); frames = [env.render()]
    for _ in range(max_steps):
        obs, r, terminated, truncated, info = env.step(int(policy(obs))); frames.append(env.render())
        if terminated or truncated: break
    ims = [Image.fromarray(f) for f in frames]
    ims[0].save(path, save_all=True, append_images=ims[1:], duration=500, loop=0)
    print(f'{len(frames) - 1} steps, crashed={info["crashed"]}, final speed={info["speed"]:.1f} m/s')
    display(IPyImage(filename=path))

## Task 1: Explore the environment (15 points)

Highway driving as an MDP: the **state** is the Kinematics observation, a 5 x 5 table with one row per vehicle (ego vehicle
first, then the 4 nearest others) and columns `[presence, x, y, vx, vy]`, all normalised. The **actions** are 5 meta-actions;
low-level controllers (a speed controller and a steering controller) turn them into acceleration and steering. The **reward**
is a weighted sum of terms you can inspect and change in the configuration.

In [ ]:
env = make_env()
obs, info = env.reset(seed=0)
cfg = env.unwrapped.config
for k in ['observation', 'action', 'lanes_count', 'vehicles_count', 'duration', 'policy_frequency', 'collision_reward',
          'right_lane_reward', 'high_speed_reward', 'reward_speed_range', 'normalize_reward']:
    print(f'{k:20s}', cfg[k])
print('\nobservation (normalised):\n', np.round(obs, 2))
print('\naction space:', env.action_space, ACTIONS)
ego_speed, others = decode_obs(obs)
print(f'\nego speed {ego_speed:.1f} m/s; others [dx, dy, dvx]:\n', np.round(others, 1))
obs, r, terminated, truncated, info = env.step(1)       # IDLE
print('\none step: reward', round(r, 3), '| info:', info)

In [ ]:
# A random driver for one episode
rng = np.random.default_rng(0)
rollout_gif(lambda obs: rng.integers(5), 'random.gif')

**Task 1 answers** (a few sentences each):

(a) What are the state, action, reward and discount factor of this MDP? Is the observation Markov?

(b) Why does highway-env use *meta-actions* (LANE_LEFT, FASTER, ...) instead of raw steering and throttle? What would change for the learning problem?

(c) Using `decode_obs`, how far ahead is the nearest vehicle in the ego lane at the start of an episode with seed 0?

(d) Read the `rewards` entry of `info`: what do the terms `collision_reward`, `right_lane_reward`, `high_speed_reward` and `on_road_reward` measure, and how are they combined into the scalar reward?

TODO

## Task 2: Rule-based baseline (15 points)

Before using RL, write a simple hand-crafted driver. Use `decode_obs(obs)` and return an action from `ACTIONS`.
Idea: if there is a vehicle ahead in our own lane (`dx > 0` and `|dy| < 2`) that is closer than `gap_m` metres, slow down;
otherwise speed up. Optionally, add lane changes (check that the target lane is free).

In [ ]:
def rule_policy(obs, gap_m=20.0):
    ego_speed, others = decode_obs(obs)
    # TODO: return 4 (SLOWER) if a vehicle in our lane is closer than gap_m metres ahead, otherwise 3 (FASTER)
    raise NotImplementedError

In [ ]:
# Compare: random, always IDLE, and your rule-based driver with three different gap thresholds
rng = np.random.default_rng(0)
results = {'random': evaluate(lambda o: rng.integers(5)), 'always IDLE': evaluate(lambda o: 1)}
for g in (10, 20, 30):
    results[f'rule, gap {g} m'] = evaluate(lambda o, g=g: rule_policy(o, g))
pd.DataFrame(results).T.round(2)

In [ ]:
# TODO: show one episode of your best rule-based driver


**Task 2 answers**: which gap threshold works best and why? How does the rule-based driver compare with the random and
IDLE drivers in crash rate and speed? What are its limitations (what situations does it handle badly)?

TODO

## Task 3: Train a DQN agent (25 points)

The agent below uses the same hyperparameters as the `dqn.json` of the rl-agents library used in the earlier version of this lab:

| rl-agents `dqn.json` | Stable-Baselines3 `DQN(...)` |
|---|---|
| layers [256, 256] | `policy_kwargs=dict(net_arch=[256, 256])` |
| gamma 0.8 | `gamma=0.8` |
| batch_size 32 | `batch_size=32` |
| memory_capacity 15000 | `buffer_size=15000` |
| target_update 50 | `target_update_interval=50` |
| epsilon-greedy, final 0.05 | `exploration_fraction=0.7`, `exploration_final_eps=0.05` |
| loss l2 | SB3 uses the Huber loss (a robust version of l2) |

Run the training cell (about 10 minutes for 15,000 steps) and look at the learning curve.

In [ ]:
TRAIN_STEPS = 15_000      # about 10 minutes on the default CPU runtime; use 4000 only for debugging

def train_dqn(config=None, steps=TRAIN_STEPS, seed=0, gamma=0.8, lane_penalty=0.0):
    env = Monitor(make_env(config, lane_penalty=lane_penalty))
    model = DQN('MlpPolicy', env, policy_kwargs=dict(net_arch=[256, 256]), learning_rate=5e-4, buffer_size=15000,
                learning_starts=200, batch_size=32, gamma=gamma, train_freq=1, gradient_steps=1,
                target_update_interval=50, exploration_fraction=0.7, exploration_final_eps=0.05,
                verbose=0, seed=seed, device='cpu')
    t0 = time.time(); model.learn(total_timesteps=steps)
    print(f'trained {steps} steps in {time.time() - t0:.0f} s, {len(env.get_episode_rewards())} episodes')
    return model, env

def plot_curve(env, label=None, window=30):
    r = np.array(env.get_episode_rewards())
    smooth = np.convolve(r, np.ones(window) / window, mode='valid') if len(r) >= window else r
    plt.plot(smooth, label=label); plt.xlabel('training episode'); plt.ylabel(f'mean return (window {window})')

dqn_policy = lambda model: (lambda obs: model.predict(obs, deterministic=True)[0])

model, train_env = train_dqn()
plt.figure(figsize=(7, 3.5)); plot_curve(train_env, 'DQN, default reward'); plt.legend(); plt.show()

In [ ]:
# Evaluate the trained agent on 30 episodes with fixed seeds and compare with Task 2
results['DQN'] = evaluate(dqn_policy(model))
pd.DataFrame(results).T.round(2)

In [ ]:
# TODO: record one episode of the trained agent


**Task 3 answers**: Does DQN beat your rule-based driver, and the random driver? Describe the behaviour you see in the GIF
(speed, lane choice, how it handles slow cars). Why is the return a noisy training signal, and what does the learning curve tell you
(is it still improving at the end)? What would you try to improve the agent (more steps, different gamma, ...)?

TODO

## Task 4: Reward shaping with a lane-change penalty (40 points)

The default reward rewards speed (`high_speed_reward` 0.4) and driving in the rightmost lane (`right_lane_reward` 0.1), and punishes
collisions (`collision_reward` -1). It does **not** punish lane changes (in this version of highway-env the config entry `lane_change_reward`
is listed but unused: you can check this in `info['rewards']`). Your task is to add that term yourself, as in the earlier version of this lab,
but with a Gymnasium wrapper instead of editing the library: `LaneChangePenalty(env, penalty)` adds `penalty` to the reward whenever the
ego vehicle starts a lane change. Pass it with `train_dqn(config, lane_penalty=...)` or `make_env(config, lane_penalty=...)`.

**Part A: warm-up in a trivial environment (10 points).** Use `config = dict(vehicles_count=0, lanes_count=2, initial_lane_id=0)`:
no other vehicles, two lanes, start in the left lane. With `lane_penalty=0` the trained vehicle should move to the right lane quickly
and stay there (use 4000 to 6000 training steps; with too few steps the agent may never have tried a lane change). Then try a penalty
that is small compared with the right-lane bonus and one that is larger than the total gain from moving right (use
`final_lane` below to see what happens), and explain why the optimal behaviour changes.

**Part B: dense traffic (20 points).** In a crowded environment (we suggest `dict(vehicles_density=1.5, lanes_count=3)`; the default traffic is easy enough to handle by braking),
train agents for **at least three** values of `lane_penalty` (for example 0, -0.1, -1, -10; it must be negative). Report for each:
crash rate, number of lane changes per episode, mean speed, episode length (use `evaluate`, 30 fixed-seed episodes). Make the penalty
very negative and look for the point at which the vehicle **prefers a collision to a lane change that would avoid it**.
Record one GIF per agent. Include a pair of configurations that differ *only* in the penalty, one with rare and one with frequent
collisions. If you cannot find such a pair, show your evidence (more episodes, more traffic, more training steps) and explain why.

**Part C: one more reward change (10 points).** Train one agent with a different modification of the reward, e.g. `collision_reward` -5 and
`high_speed_reward` 0.1 (safe), or `high_speed_reward` 1.0 (fast), or `right_lane_reward` 1.0 (keep right).

**Important:** returns of agents trained with different rewards are not comparable. Always evaluate with the *same* behaviour metrics
(crash rate, speed, lane changes), measured in the same environment (`evaluate` does not add the penalty).

In [ ]:
# Part A: trivial environment. TODO: fill in the penalties you want to test (at least 0 and one more)
simple = dict(vehicles_count=0, lanes_count=2, initial_lane_id=0)
def final_lane(model, n=10):          # lane index at the end of the episode (0 = leftmost, 1 = rightmost)
    env = make_env(simple); out = []
    for ep in range(n):
        obs, _ = env.reset(seed=ep); done = False
        while not done:
            obs, r, te, tr, info = env.step(int(model.predict(obs, deterministic=True)[0])); done = te or tr
        out.append(env.unwrapped.vehicle.lane_index[2])
    return np.mean(out)
simple_agents = {}


In [ ]:
# Part B: dense traffic. TODO: choose the environment and at least three penalties (negative numbers)
dense = {}                      # e.g. dict(vehicles_density=1.5, lanes_count=3)
penalties = []                  # e.g. [0.0, -0.1, -1.0, -10.0]
agents = {}


In [ ]:
# Behaviour metrics in the SAME environment (without the penalty) for all agents, 30 fixed-seed episodes
# TODO: build the table (crash_rate, lane_changes, speed, length per agent) with evaluate(dqn_policy(m), 30, dense)


In [ ]:
# TODO: record one GIF per agent, in particular for the two configurations that differ only in the penalty


In [ ]:
# Part C: one more reward change. TODO: choose a config and train
other = {}                      # e.g. {'collision_reward': -5, 'high_speed_reward': 0.1}


**Task 4 answers**:

(A) Which behaviour did you see in the trivial environment for each penalty, and why? What is the smallest penalty (per lane change) for which staying in the left lane becomes optimal, in terms of the discounted right-lane bonus?

(B) How did the number of lane changes and the crash rate change with the penalty? Which pair of configurations differs only in the penalty, with rare versus frequent collisions? At what penalty does the vehicle prefer a collision to a lane change, and why does a *large* penalty not simply make the agent brake instead? (Think about what braking does to the speed reward and what else is on the road.)

(C) Did the extra reward change alter the behaviour as you expected? Name one way the agent could exploit the reward, or one reason why a change had little effect.

(D) Why is it wrong to compare the returns of agents trained with different rewards? How would you design a reward for a safe but comfortable driver (few lane changes, but no collisions)? Why is it a problem that results come from a single training seed with 30 evaluation episodes?

TODO

## Submission checklist

- Notebook executed, all outputs and learning curves visible, sharing set to *anyone with the link*
- PDF report (at most 5 pages) with the tables, figures (curves, one GIF frame per agent) and written answers of Tasks 1 to 4
- Training takes a while: start Tasks 3 and 4 early, and keep the browser tab open (Colab disconnects idle sessions)